# CS-681 Project: Text Evaluation
Zero-shot vs structured template prompting (MedGemma).

## 1. Load the reports

In [ ]:
import glob, re, os
import numpy as np
import pandas as pd

# Find reports.csv anywhere under the attached inputs
paths = glob.glob("/kaggle/input/**/reports.csv", recursive=True)
print("Found:", paths)
df = pd.read_csv(paths[0])
print(df.shape)
print(df.groupby("prompt_type").size())

## 2. Extract the predicted grade from each report

In [ ]:
GRADE_WORDS = [  # (regex, grade)
    (r"\bproliferative\b", 4), (r"\bsevere\b", 3), (r"\bmoderate\b", 2), (r"\bmild\b", 1),
    (r"\bno (?:obvious |apparent |clear |definite |significant |visible )*(?:signs? of |evidence of )?(?:diabetic retinopathy|dr)\b", 0),
    (r"\bnormal (?:retina|fundus)\b", 0)]
CONTEXT = r"(retinopathy|npdr|\bdr\b|severity|stage|grade)"

def grade_structured(text):
    '''Read the number after 'SEVERITY GRADE:'; NaN if missing.'''
    m = re.search(r"SEVERITY GRADE:\s*([0-4])", str(text))
    return int(m.group(1)) if m else np.nan

def grade_zero_shot(text):
    '''Rule-based extraction from free text.
    Returns 0-4, or NaN when the report gives a range ('mild to moderate'), conflicting grades, or no grade.'''
    t = str(text).lower().replace("**", "").replace("*", "")
    t = re.sub(r"non[- ]?proliferative", "npdr", t)  # so 'proliferative' only matches PDR
    # Prefer the text after the last heading that mentions assessment/severity
    heads = [m.end() for m in re.finditer(r"\n[^\n]*(assessment|severity)[^\n]*:\s*\n", t)]
    section = t[heads[-1]:] if heads else t
    section = re.split(r"\n[^\n]*(important considerations|disclaimer|recommendation|next steps)[^\n]*:", section)[0]
    # First sentence that talks about retinopathy/severity and names a grade decides
    for sent in re.split(r"(?<=[.!?\n])\s+", section):
        if not re.search(CONTEXT, sent):
            continue
        found = sorted({g for p, g in GRADE_WORDS if re.search(p, sent)})
        if found:
            return found[0] if len(found) == 1 else np.nan
    return np.nan

df["pred_grade"] = np.where(df["prompt_type"] == "structured",
                            df["report"].apply(grade_structured),
                            df["report"].apply(grade_zero_shot))

# Extraction rate = share of reports with one clear grade
print(df.groupby("prompt_type")["pred_grade"].apply(lambda s: s.notna().mean()).rename("extraction_rate"))

## 3. Grading metrics per prompt

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

rows = []
for name, g in df.groupby("prompt_type"):
    ok = g.dropna(subset=["pred_grade"])
    y, p = ok["true_grade"].astype(int), ok["pred_grade"].astype(int)
    ref_y, ref_p = (y >= 2), (p >= 2)  # referable DR = grade 2 or higher
    tp = (ref_y & ref_p).sum(); fn = (ref_y & ~ref_p).sum()
    tn = (~ref_y & ~ref_p).sum(); fp = (~ref_y & ref_p).sum()
    rows.append({
        "prompt": name,
        "extraction_rate": round(len(ok) / len(g), 3),
        # Strict accuracy counts unextracted reports as wrong
        "accuracy_strict": round((p.values == y.values).sum() / len(g), 3),
        "accuracy_extracted": round(accuracy_score(y, p), 3),
        "within_1_grade": round((abs(p - y) <= 1).mean(), 3),
        "QWK": round(cohen_kappa_score(y, p, weights="quadratic"), 3),
        "referable_sensitivity": round(tp / (tp + fn), 3),
        "referable_specificity": round(tn / (tn + fp), 3) if (tn + fp) else np.nan,
        "avg_seconds": round(g["seconds"].mean(), 1),
        "avg_words": round(g["report"].str.split().str.len().mean(), 1),
    })
metrics = pd.DataFrame(rows).set_index("prompt").T
print(metrics)
metrics.to_csv("/kaggle/working/text_metrics.csv")

## 4. Confusion matrices (figures for the paper)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, g) in zip(axes, df.groupby("prompt_type")):
    ok = g.dropna(subset=["pred_grade"])
    cm = confusion_matrix(ok["true_grade"], ok["pred_grade"].astype(int), labels=range(5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=range(5), yticklabels=range(5))
    ax.set_title(f"{name} (n={len(ok)})")
    ax.set_xlabel("Predicted grade"); ax.set_ylabel("True grade")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrices.png", dpi=200)
plt.show()

## 5. Structured reports: format compliance and internal consistency

In [ ]:
FIELDS = ["Microaneurysms", "Hemorrhages", "Hard exudates", "Cotton wool spots",
          "Venous beading or IRMA", "Neovascularization", "Vitreous or preretinal hemorrhage"]
SECTIONS = ["IMAGE QUALITY:", "FINDINGS:", "SEVERITY GRADE:", "SEVERITY LABEL:",
            "RECOMMENDATION:", "SUMMARY:"]

def parse_findings(text):
    """Return {finding: True/False/None} from the FINDINGS block."""
    out = {}
    for f in FIELDS:
        m = re.search(rf"-\s*{re.escape(f)}:\s*(Present|Absent)", str(text), re.I)
        out[f] = None if not m else m.group(1).lower() == "present"
    return out

def get_summary(text):
    m = re.search(r"SUMMARY:\s*(.+)", str(text), re.S)
    return m.group(1).strip() if m else ""

def check(row):
    text, g = row["report"], row["pred_grade"]
    f = parse_findings(text)
    any_present = any(v for v in f.values() if v is not None)
    pdr_signs = bool(f["Neovascularization"] or f["Vitreous or preretinal hemorrhage"])
    only_ma = f["Microaneurysms"] and not any(v for k, v in f.items() if k != "Microaneurysms" and v)
    rules = {
        "grade0_has_findings": g == 0 and any_present,
        "grade1_not_ma_only": g == 1 and not only_ma,
        "grade_ge1_no_findings": g >= 1 and not any_present,
        "grade4_without_pdr_signs": g == 4 and not pdr_signs,
        "pdr_signs_but_below4": pdr_signs and g < 4,
    }
    # Summary mentions a finding that the FINDINGS list marked Absent
    summ = get_summary(text).lower()
    keywords = {"Neovascularization": "neovascular", "Hemorrhages": "hemorrhage",
                "Hard exudates": "exudate", "Cotton wool spots": "cotton wool",
                "Venous beading or IRMA": "venous beading", "Microaneurysms": "microaneurysm"}
    rules["summary_contradicts_findings"] = any(
        f[k] is False and kw in summ for k, kw in keywords.items())
    return pd.Series({
        "format_complete": all(s in str(text) for s in SECTIONS) and None not in f.values(),
        **rules,
        "consistent": not any(rules.values()),
    })

st = df[df["prompt_type"] == "structured"].copy()
st = pd.concat([st, st.apply(check, axis=1)], axis=1)
consistency = st.drop(columns=["image", "true_grade", "prompt_type", "report",
                               "seconds", "pred_grade"]).mean().round(3)
print(consistency)
consistency.to_csv("/kaggle/working/structured_consistency.csv")

## 6. Export summaries for the audio part and 30 reports for human rating

In [ ]:
# Summaries for SpeechT5 (structured prompt only)
st["summary"] = st["report"].apply(get_summary)
st[["image", "true_grade", "pred_grade", "summary"]].to_csv(
    "/kaggle/working/summaries_for_audio.csv", index=False)

# 30 images (6 per grade), both prompt versions side by side, for blind human rating
pick = df[df["prompt_type"] == "structured"].groupby("true_grade").sample(n=6, random_state=42)["image"]
human = df[df["image"].isin(pick)].pivot(index="image", columns="prompt_type", values="report").reset_index()
for col in ["clarity_zero_shot", "relevance_zero_shot", "hallucination_zero_shot",
            "clarity_structured", "relevance_structured", "hallucination_structured"]:
    human[col] = ""  # to be filled with scores 1-5 by each rater
human.to_csv("/kaggle/working/human_rating_30.csv", index=False)

df.to_csv("/kaggle/working/reports_with_grades.csv", index=False)
print("Saved:", os.listdir("/kaggle/working"))